# HDD 02 — Prepare the research dataset
This notebook is independent of Notebook 01.

**Inputs:** the exported `hdd_pilot_q1_2021.csv`, Q1 and Q2 2021 Backblaze ZIPs.
**Outputs:** versioned settings, audited daily records, separate evaluation labels, drive splits, raw windows, train-fitted preprocessing, and a sealed test directory.

This is a manageable **single-model experiment**, not the complete multi-model study promised in the proposal. Retain Notebook 01 as exploratory evidence. No real failure labels are used to train, calibrate, choose models, or select drives.

Use a CPU runtime here. Downloads and CSV scanning dominate runtime. Do not run Notebook 03 until the completion cell passes. No real Backblaze outputs are precomputed in this supplied notebook.


## 1. Install dependencies
Run once in Colab. On a local Jupyter installation, this installs into the notebook kernel. Restart the kernel if it reports that an already loaded package needs restarting.


In [1]:
%pip -q install "numpy>=1.26,<3" "pandas>=2.2,<3" "scikit-learn>=1.4,<2" "pyarrow>=16" "requests>=2.31" "tqdm>=4.66" joblib matplotlib


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Choose a persistent component folder
**Colab:** first run the optional Drive cell below, then set `COMPONENT_ROOT` to a folder in mounted Drive. You can later copy the small outputs into your Git checkout. A folder under `/content` disappears when the runtime resets.

**Local Jupyter:** set it to the absolute path of your existing `01_component_1_behavioral_drift_genai` folder. This notebook creates missing subfolders, not a Git repository. It does not overwrite your existing YAML files.

Put the CSV exported from Notebook 01 in `data/interim/hdd_pilot_q1_2021.csv`. If you did not download it, rerun Notebook 01 through its pilot CSV export first.


In [2]:
# Colab only: uncomment both lines to mount your own Drive.
# from google.colab import drive
# drive.mount('/content/drive')


In [2]:
from pathlib import Path
import json, hashlib, platform, re, zipfile, warnings
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import requests, sklearn, joblib
from tqdm.auto import tqdm

# EDIT THIS ONE PATH. Local Windows example: Path(r"D:\research\J26-DS-342\01_component_1_behavioral_drift_genai")
from pathlib import Path

COMPONENT_NAME = "01_component_1_behavioral_drift_genai"

COMPONENT_ROOT = None

for folder in [Path.cwd(), *Path.cwd().parents]:
    if folder.name == COMPONENT_NAME:
        COMPONENT_ROOT = folder
        break

    candidate = folder / COMPONENT_NAME

    if candidate.is_dir():
        COMPONENT_ROOT = candidate
        break

if COMPONENT_ROOT is None:
    raise FileNotFoundError(
        "Could not find the component folder. "
        "Open your shared repository folder in VS Code."
    )

print("Component folder:", COMPONENT_ROOT)
if str(COMPONENT_ROOT).startswith('/content/drive/') and not Path('/content/drive/MyDrive').exists():
    raise RuntimeError("Mount Google Drive above, or edit COMPONENT_ROOT for your local checkout.")
RUN_ID = 'single_model_q1q2_2021_v1'
DIR = {name: COMPONENT_ROOT / name for name in [
    'config', 'data/raw', 'data/interim', 'data/processed', 'data/splits',
    'data/features', 'models/trained', 'results', 'figures', 'experiments']}
for folder in DIR.values():
    folder.mkdir(parents=True, exist_ok=True)
RUN = DIR['experiments'] / RUN_ID
RUN.mkdir(exist_ok=True)
PILOT_CSV = DIR['data/interim'] / 'hdd_pilot_q1_2021.csv'
if not PILOT_CSV.exists():
    raise FileNotFoundError(f"Copy the exported Notebook 01 pilot CSV to: {PILOT_CSV}")
print('Component:', COMPONENT_ROOT)
print('Experiment:', RUN_ID)


Component folder: d:\SLIIT\Y4S1\RP\Research\J26-DS-342-\01_component_1_behavioral_drift_genai
Component: d:\SLIIT\Y4S1\RP\Research\J26-DS-342-\01_component_1_behavioral_drift_genai
Experiment: single_model_q1q2_2021_v1


## 3. Validate the old pilot and reproduce its serial-number split
We reproduce Notebook 01's exact 70/15/15 split from its sorted pilot serials and NumPy seed 42. This avoids accidentally moving the already inspected development drive into the final test.

The pilot CSV is the authoritative list of the previous 1,000 serials. A content hash is recorded. No drive is selected using its failure outcome.


In [3]:
MODEL = 'ST12000NM0008'
SEED = 42
FEATURES = ['smart_5_raw', 'smart_9_raw', 'smart_194_raw', 'smart_197_raw', 'smart_198_raw']
WINDOW_DAYS = 30
TEST_DRIVES = 150

def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024*1024), b''):
            digest.update(block)
    return digest.hexdigest()

pilot = pd.read_csv(PILOT_CSV, dtype={'serial_number':str, 'model':str}, parse_dates=['date'])
required = ['date', 'serial_number', 'model'] + FEATURES
assert set(required).issubset(pilot.columns), 'Pilot CSV has missing required columns.'
assert pilot['serial_number'].notna().all() and pilot['date'].notna().all()
assert pilot['model'].eq(MODEL).all(), 'Unexpected model in the pilot.'
assert not pilot.duplicated(['serial_number','date']).any()
assert pilot['date'].between('2021-01-01','2021-03-31').all()
pilot_ids = np.array(sorted(pilot['serial_number'].unique()))
assert len(pilot_ids) == 1000, 'Expected the original 1,000-drive pilot; do not silently substitute a new sample.'
shuffled = pilot_ids.copy()
np.random.default_rng(SEED).shuffle(shuffled)
split_ids = {
    'train': sorted(shuffled[:700].tolist()),
    'calibration': sorted(shuffled[700:850].tolist()),
    'development': sorted(shuffled[850:].tolist()),
}
print({key:len(ids) for key,ids in split_ids.items()})
print('Pilot rows:',len(pilot), '| SHA256:',sha256(PILOT_CSV))


{'train': 700, 'calibration': 150, 'development': 150}
Pilot rows: 89896 | SHA256: 055254bb1e689d8be597f62daceb32926eeba815e41a4b2af6900f7650b97112


## 4. Download or reuse the original archives
The archives are large. Reuse the Q1 ZIP already downloaded by copying it to `data/raw/data_Q1_2021.zip`. 

Downloads use a temporary file and rename only after a ZIP integrity check. Existing corrupt files cause an explicit error rather than silent use. Never commit the archives to normal Git tracking.


In [4]:
ARCHIVES = {
    'Q1': DIR['data/raw'] / 'data_Q1_2021.zip',
    'Q2': DIR['data/raw'] / 'data_Q2_2021.zip',
}
BASE_URL = 'https://f001.backblazeb2.com/file/Backblaze-Hard-Drive-Data/'

def get_archive(path):
    if path.exists():
        if not zipfile.is_zipfile(path):
            raise ValueError(f'Invalid archive; inspect or replace manually: {path}')
        print('Reusing',path.name)
    else:
        temporary = path.with_suffix('.zip.part')
        try:
            with requests.get(BASE_URL + path.name, stream=True, timeout=(30,180)) as response:
                response.raise_for_status()
                with open(temporary,'wb') as output, tqdm(
                    total=int(response.headers.get('content-length',0)) or None,
                    unit='B',unit_scale=True,desc=path.name) as progress:
                    for block in response.iter_content(1024*1024):
                        if block:
                            output.write(block)
                            progress.update(len(block))
            if not zipfile.is_zipfile(temporary):
                raise ValueError('Response was not a ZIP archive.')
            with zipfile.ZipFile(temporary) as archive:
                bad = archive.testzip()
                if bad is not None:
                    raise ValueError(f'CRC failure in {bad}')
            temporary.replace(path)
        except Exception:
            temporary.unlink(missing_ok=True)
            raise
    return path

for path in ARCHIVES.values():
    get_archive(path)

def daily_members(archive):
    members = sorted(name for name in archive.namelist()
        if '__MACOSX' not in name and re.fullmatch(r'\d{4}-\d{2}-\d{2}\.csv',Path(name).name))
    dates = [pd.Timestamp(Path(name).stem) for name in members]
    assert len(dates) == len(set(dates)), 'Duplicate calendar dates inside archive.'
    return members

for quarter,path in ARCHIVES.items():
    with zipfile.ZipFile(path) as archive:
        names = daily_members(archive)
        expected = pd.date_range('2021-01-01','2021-03-31') if quarter=='Q1' else pd.date_range('2021-04-01','2021-06-30')
        assert [pd.Timestamp(Path(n).stem) for n in names] == list(expected), 'Unexpected archive date coverage.'
        print(quarter,len(names),'days')


data_Q1_2021.zip: 100%|██████████| 890M/890M [01:55<00:00, 7.68MB/s] 
data_Q2_2021.zip: 100%|██████████| 908M/908M [01:51<00:00, 8.13MB/s] 


Q1 90 days
Q2 91 days


## 5. Select the sealed-test serials without labels
Choose 150 other drives of the same model that were present on 1 April. These serials must not occur in the original pilot. They will only be **scored in June**.

This test estimates performance for drives present on 1 April, not all drives entering the fleet later. Missing coverage and failures can reduce the scored group; report that attrition rather than replacing inconvenient drives.


In [5]:
with zipfile.ZipFile(ARCHIVES['Q2']) as archive:
    with archive.open(daily_members(archive)[0]) as handle:
        april_first = pd.read_csv(handle, usecols=['serial_number','model'], dtype=str)
candidates = sorted(set(april_first.loc[april_first['model'].eq(MODEL),'serial_number'].dropna()) - set(pilot_ids))
if len(candidates) < TEST_DRIVES:
    raise ValueError(f'Only {len(candidates)} eligible non-pilot test drives; revise and version the protocol explicitly.')
split_ids['test'] = sorted(np.random.default_rng(SEED+1).choice(candidates,size=TEST_DRIVES,replace=False).tolist())
for a in split_ids:
    for b in split_ids:
        if a != b:
            assert set(split_ids[a]).isdisjoint(split_ids[b])

PERIODS = {
    'train': ['2021-01-30','2021-03-31'],
    'calibration': ['2021-04-01','2021-04-30'],
    'development': ['2021-05-01','2021-05-31'],
    'test': ['2021-06-01','2021-06-30'],
}
config = {
    'run_id':RUN_ID,'model':MODEL,'seed':SEED,'test_selection_seed':SEED+1,
    'raw_features':FEATURES,'window_days':WINDOW_DAYS,'stride_days':1,
    'scoring_endpoint_periods':PERIODS,'pilot_sha256':sha256(PILOT_CSV),
    'missing_rule':'strict: reject any window with missing/nonfinite/negative SMART values; retain masks and audit',
    'history_rule':'allow earlier context within the same held-out drive; never use future rows',
    'labels':'failure stored separately; forbidden for fitting and development selection',
    'test_population':'same-model non-pilot drives present 2021-04-01; June scoring only',
    'versions':{'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__,'sklearn':sklearn.__version__},
}
CONFIG_PATH = DIR['config'] / f'{RUN_ID}.json'
SPLIT_PATH = DIR['data/splits'] / f'{RUN_ID}_serials.json'
for path,value in [(CONFIG_PATH,config),(SPLIT_PATH,split_ids)]:
    if path.exists() and json.loads(path.read_text()) != value:
        raise RuntimeError(f'Existing settings differ: {path}. Use a NEW RUN_ID rather than overwrite an experiment.')
    path.write_text(json.dumps(value,indent=2))
print('Saved fixed settings and serial assignments.')


Saved fixed settings and serial assignments.


## 6. Extract telemetry and keep failure labels separate
Only sampled serials are retained. All selected Q1/Q2 rows are read once per archive, in chunks. `failure` is removed from the telemetry table immediately.

The extraction creates a technical failure table but does not print outcomes. A failure flag is an evaluation outcome, not an input feature. Do not interpret disappearance from the dataset as a failure.

A full-cohort Q2 audit below examines coverage and SMART variation **for development serials only**. June test telemetry and all failure outcomes remain out of exploratory displays. File-level validation is permitted; scientific inspection of the test must wait until the final pipeline is frozen.


In [6]:
ALL_IDS = set().union(*(set(ids) for ids in split_ids.values()))
columns = ['date','serial_number','model','failure'] + FEATURES
telemetry_parts, label_parts = [], []
for quarter,path in ARCHIVES.items():
    with zipfile.ZipFile(path) as archive:
        for name in tqdm(daily_members(archive),desc=f'Extracting {quarter}'):
            with archive.open(name) as handle:
                for chunk in pd.read_csv(handle, usecols=columns,
                    dtype={'serial_number':str,'model':str},chunksize=50000):
                    selected = chunk.loc[chunk['model'].eq(MODEL) & chunk['serial_number'].isin(ALL_IDS)].copy()
                    if selected.empty:
                        continue
                    selected['date'] = pd.to_datetime(selected['date'],errors='raise')
                    assert selected['date'].eq(pd.Timestamp(Path(name).stem)).all()
                    failure = pd.to_numeric(selected.pop('failure'),errors='raise')
                    assert failure.notna().all() and failure.isin([0,1]).all(), 'Unexpected failure encoding.'
                    labels = selected[['date','serial_number']].copy()
                    labels['failure'] = failure.astype('int8')
                    label_parts.append(labels)
                    telemetry_parts.append(selected)
if not telemetry_parts:
    raise ValueError('No selected records found.')
daily = pd.concat(telemetry_parts,ignore_index=True).sort_values(['serial_number','date']).reset_index(drop=True)
labels = pd.concat(label_parts,ignore_index=True).sort_values(['serial_number','date']).reset_index(drop=True)
assert not daily.duplicated(['serial_number','date']).any()
assert not labels.duplicated(['serial_number','date']).any()
assert daily[['date','serial_number']].equals(labels[['date','serial_number']])
assert 'failure' not in daily.columns
assert set(daily['serial_number']).issubset(ALL_IDS)
for feature in FEATURES:
    daily[feature] = pd.to_numeric(daily[feature],errors='coerce')

# Verify that extraction reproduces the original pilot's Q1 input.
q1_pilot = daily.loc[daily['serial_number'].isin(pilot_ids) & daily['date'].le('2021-03-31'),required]
a = pilot[required].sort_values(['serial_number','date']).reset_index(drop=True)
b = q1_pilot.sort_values(['serial_number','date']).reset_index(drop=True)
pd.testing.assert_frame_equal(a,b,check_dtype=False)
print('Extracted records match the original Q1 pilot. Failure outcomes are not displayed.')
del telemetry_parts, label_parts, april_first


Extracting Q2: 100%|██████████| 91/91 [01:24<00:00,  1.07it/s]


Extracted records match the original Q1 pilot. Failure outcomes are not displayed.


## 7. Save outcome tables behind an explicit evaluation boundary
The complete extraction spans Q1–Q2 for the selected serials. Failure-free observation through the archive end does **not** prove long-term health. For 7/14/30-day warning evaluation, score only endpoints with enough future observation, or mark their outcomes unknown/censored. Extending follow-up into Q3 is a later evaluation task.

Notebook 03 never opens `evaluation_only`. If you inspect labels to revise data selection or model choices, those data become development data and a new blind test is required.


In [7]:
EVAL_DIR = DIR['data/processed'] / RUN_ID / 'evaluation_only'
EVAL_DIR.mkdir(parents=True,exist_ok=True)
labels.to_parquet(EVAL_DIR/'failure_observations.parquet',index=False)
labels.groupby('serial_number')['date'].agg(first_observed='min',last_observed='max').to_csv(EVAL_DIR/'followup_coverage.csv')
(EVAL_DIR/'README.txt').write_text(
    'EVALUATION ONLY. Do not read for feature preparation, training, calibration, clustering, drift updates, or model selection. '
    'failure=0 is only a daily observation. Absence is not failure. Apply follow-up censoring rules. '
    'A later final-evaluation notebook may open these data only after the pipeline is frozen.\n')
del labels
print('Separate evaluation files saved; outcomes remain unopened.')


Separate evaluation files saved; outcomes remain unopened.


## 8. Audit the non-test data
SMART 9 changes of 16/32 in Notebook 01 should not be assumed to be literal daily operating hours without confirming the model's encoding. Keep it labelled as a raw attribute for now. Error counters can be constant and sparse. A zero is an observed reading, not a missing value.

Negative or nonfinite readings are marked unusable for these five nonnegative raw fields. No arbitrary upper-temperature cutoff is used; extreme values may be meaningful and need separate investigation.


In [8]:
non_test = daily.loc[~daily['serial_number'].isin(split_ids['test']) & daily['date'].le('2021-05-31')].copy()
audit_rows=[]
for split in ['train','calibration','development']:
    start,end=PERIODS[split]
    subset=non_test.loc[non_test['serial_number'].isin(split_ids[split]) & non_test['date'].between(start,end)]
    for feature in FEATURES:
        values=subset[feature]
        audit_rows.append({'split':split,'feature':feature,'rows':len(subset),
            'missing':int(values.isna().sum()),'negative':int(values.lt(0).sum()),
            'nonfinite':int((~np.isfinite(values.to_numpy(dtype=float))).sum()),
            'nonzero':int((values.notna() & values.ne(0)).sum()),
            'minimum':values.min(),'maximum':values.max()})
feature_audit=pd.DataFrame(audit_rows)
display(feature_audit)
feature_audit.to_csv(RUN/'feature_audit.csv',index=False)
coverage=non_test.groupby('serial_number')['date'].agg(first_date='min',last_date='max',observed_days='nunique')
coverage.to_csv(RUN/'non_test_coverage.csv')
print('June and test-drive telemetry excluded from these displays.')


,split,feature,rows,missing,negative,nonfinite,nonzero,minimum,maximum
0,train,smart_5_raw,42689,0,0,0,485,0.0,416.0
1,train,smart_9_raw,42689,0,0,0,42689,1268.0,12944.0
2,train,smart_194_raw,42689,0,0,0,42689,17.0,53.0
3,train,smart_197_raw,42689,0,0,0,86,0.0,344.0
4,train,smart_198_raw,42689,0,0,0,86,0.0,344.0
5,calibration,smart_5_raw,4410,0,0,0,150,0.0,8960.0
6,calibration,smart_9_raw,4410,0,0,0,4410,2848.0,13665.0
7,calibration,smart_194_raw,4410,0,0,0,4410,18.0,46.0
8,calibration,smart_197_raw,4410,0,0,0,30,0.0,24.0
9,calibration,smart_198_raw,4410,0,0,0,30,0.0,24.0


June and test-drive telemetry excluded from these displays.


## 9. Build daily calendars and masks
Insert only internal calendar gaps between each drive's first and last observations. Do not invent observations after removal. Preserve original raw values; the window builder rejects invalid values rather than imputing them.

The first model uses fully observed windows. Later masked self-supervised work can explicitly support partial windows; use a new experiment version when changing eligibility rules. Compare excluded and included drives when reporting results.


In [9]:
def prepare_daily(frame,features):
    pieces=[]
    for serial,history in frame.groupby('serial_number',sort=True):
        history=history.sort_values('date').set_index('date').copy()
        assert not history.index.duplicated().any()
        model=history['model'].iloc[0]
        history['record_present']=True
        history=history.reindex(pd.date_range(history.index.min(),history.index.max(),freq='D'))
        history.index.name='date'
        history['serial_number']=serial
        history['model']=model
        history['record_present']=history['record_present'].eq(True)
        for feature in features:
            values=history[feature].to_numpy(dtype=float)
            history[feature+'_observed']=history[feature].notna()
            history[feature+'_usable']=np.isfinite(values) & (values>=0)
        pieces.append(history.reset_index())
    if not pieces:
        raise ValueError('No daily records available.')
    return pd.concat(pieces,ignore_index=True).sort_values(['serial_number','date']).reset_index(drop=True)

prepared=prepare_daily(daily,FEATURES)
assert 'failure' not in prepared.columns
PROCESSED = DIR['data/processed']/RUN_ID
PROCESSED.mkdir(exist_ok=True)
# Keep the test serials physically separate from working data.
for split,ids in split_ids.items():
    folder=PROCESSED/('sealed_test' if split=='test' else split)
    folder.mkdir(exist_ok=True)
    prepared.loc[prepared['serial_number'].isin(ids)].to_parquet(folder/'daily.parquet',index=False)
print('Daily records and original-observation/usable masks saved by split.')


Daily records and original-observation/usable masks saved by split.


## 10. Create daily 30-day windows
**The split dates refer to scoring endpoints.** An April 1 calibration window uses March 3–April 1 history from that calibration drive. No training drive appears there and no future day is used. This is an online scoring setup with historical context, not a requirement that every context date be later than the training cutoff.

This yields up to 30 April calibration windows per covered drive, rather than the two highly overlapping March windows in the original pilot. These windows are still dependent. Later confidence intervals should resample drives, not individual windows.

The test uses June endpoints and its own earlier context. No test examples or distributions are printed.


In [10]:
def build_windows(frame, ids, start, end, features, window_days):
    arrays,masks,metadata=[],[],[]
    counts={'expected_endpoints':len(ids)*len(pd.date_range(start,end)),
            'no_record_at_endpoint':0,'insufficient_history':0,'invalid_or_missing_window':0,'accepted_windows':0}
    endpoints=pd.date_range(start,end)
    grouped={serial:part.sort_values('date').set_index('date') for serial,part in frame.groupby('serial_number')}
    for serial in sorted(ids):
        history=grouped.get(serial)
        for endpoint in endpoints:
            if history is None or endpoint not in history.index or not bool(history.at[endpoint,'record_present']):
                counts['no_record_at_endpoint']+=1
                continue
            dates=pd.date_range(end=endpoint,periods=window_days)
            if dates[0]<history.index.min():
                counts['insufficient_history']+=1
                continue
            block=history.reindex(dates)
            values=block[features].to_numpy(dtype=np.float32)
            observed=block[[f+'_observed' for f in features]].eq(True).to_numpy(dtype=bool)
            if not np.isfinite(values).all() or (values<0).any() or not observed.all():
                counts['invalid_or_missing_window']+=1
                continue
            arrays.append(values)
            masks.append(observed)
            metadata.append({'serial_number':serial,'start_date':dates[0],'end_date':endpoint})
    counts['accepted_windows']=len(arrays)
    assert sum(counts[k] for k in ['no_record_at_endpoint','insufficient_history','invalid_or_missing_window','accepted_windows'])==counts['expected_endpoints']
    X=np.stack(arrays) if arrays else np.empty((0,window_days,len(features)),dtype=np.float32)
    M=np.stack(masks) if masks else np.empty_like(X,dtype=bool)
    meta=pd.DataFrame(metadata,columns=['serial_number','start_date','end_date'])
    return X,M,meta,counts

FEATURE_ROOT = DIR['data/features']/RUN_ID
FEATURE_ROOT.mkdir(exist_ok=True)
window_audits=[]
for split,ids in split_ids.items():
    X,M,meta,counts=build_windows(prepared,ids,*PERIODS[split],FEATURES,WINDOW_DAYS)
    if len(X)==0:
        raise ValueError(f'No valid windows for {split}; inspect inputs and coverage before proceeding.')
    folder=FEATURE_ROOT/('sealed_test' if split=='test' else split)
    folder.mkdir(exist_ok=True)
    np.savez_compressed(folder/'windows_raw.npz',X=X,observed_mask=M)
    meta.to_parquet(folder/'metadata.parquet',index=False)
    (folder/'window_audit.json').write_text(json.dumps(counts,indent=2))
    if split!='test':
        window_audits.append({'split':split,**counts,'drives_scored':meta['serial_number'].nunique()})
        print(split,X.shape,'| drives:',meta['serial_number'].nunique())
    else:
        print('Sealed test arrays saved; detailed audit retained inside sealed_test.')
window_audit=pd.DataFrame(window_audits)
window_audit.to_csv(RUN/'window_audit.csv',index=False)
display(window_audit)
del X,M,meta,prepared,daily,non_test


train (42578, 30, 5) | drives: 700
calibration (4410, 30, 5) | drives: 147
development (4642, 30, 5) | drives: 150
Sealed test arrays saved; detailed audit retained inside sealed_test.


,split,expected_endpoints,no_record_at_endpoint,insufficient_history,invalid_or_missing_window,accepted_windows,drives_scored
0,train,42700,11,0,111,42578,700
1,calibration,4500,90,0,0,4410,147
2,development,4650,0,0,8,4642,150


## 11. Fit daily preprocessing on training records only
Fit on unique observed training days, not repeated copies of the same day inside overlapping windows. Apply `log1p` to the four nonnegative counter/usage fields and use the training median/IQR per attribute. If IQR is zero, keep a scale of 1; do not divide by zero or discard a rare counter.

This daily preprocessing is for neural sequence models. Notebook 03 separately fits its summary-feature scaler on training summaries for the tabular baselines. The masks remain distinct from scaled numerical values.


In [11]:
LOG_FEATURES=['smart_5_raw','smart_9_raw','smart_197_raw','smart_198_raw']

def log_values(X,features=FEATURES,log_features=LOG_FEATURES):
    values=np.array(X,dtype=np.float64,copy=True)
    if not np.isfinite(values).all() or (values<0).any():
        raise ValueError('Expected complete nonnegative raw values.')
    for feature in log_features:
        values[...,features.index(feature)]=np.log1p(values[...,features.index(feature)])
    return values

training_daily=pd.read_parquet(PROCESSED/'train'/'daily.parquet')
training_daily=training_daily.loc[training_daily['date'].le(PERIODS['train'][1])]
fit_values=training_daily[FEATURES].to_numpy(dtype=float)
# A row must be fully usable; no invalid or future days affect fitted statistics.
valid=np.isfinite(fit_values).all(axis=1) & (fit_values>=0).all(axis=1)
fit_values=log_values(fit_values[valid])
assert len(fit_values)>0
center=np.median(fit_values,axis=0)
iqr=np.quantile(fit_values,.75,axis=0)-np.quantile(fit_values,.25,axis=0)
scale=np.where(iqr>1e-12,iqr,1.0)
preprocessing={'features':FEATURES,'log_features':LOG_FEATURES,'center':center.tolist(),
    'scale':scale.tolist(),'original_iqr':iqr.tolist(),'fit_rows':len(fit_values),
    'fit_end':PERIODS['train'][1],'fit_split':'train','zero_iqr_rule':'scale=1'}
PREPROCESS_PATH=DIR['models/trained']/f'{RUN_ID}_daily_preprocessing.json'
PREPROCESS_PATH.write_text(json.dumps(preprocessing,indent=2))
for split in split_ids:
    folder=FEATURE_ROOT/('sealed_test' if split=='test' else split)
    with np.load(folder/'windows_raw.npz') as data:
        transformed=(log_values(data['X'])-center)/scale
        assert np.isfinite(transformed).all()
        np.savez_compressed(folder/'windows_scaled.npz',X=transformed.astype(np.float32),observed_mask=data['observed_mask'])
print('Frozen training-only preprocessing saved and applied.')
del training_daily,fit_values,transformed


Frozen training-only preprocessing saved and applied.


## 12. Verify saved files and record provenance
These are mechanical checks, including the sealed test's schema and dates. They do not inspect its outcome labels, anomaly scores or feature distributions. SHA256 checksums detect accidental changes; ZIP CRC checks do not establish an independent publisher-supplied checksum.

The final completion JSON is required by Notebook 03. If this cell fails, fix the cause rather than commenting out its assertions.


In [12]:
checks=[]
for split,ids in split_ids.items():
    folder=FEATURE_ROOT/('sealed_test' if split=='test' else split)
    metadata=pd.read_parquet(folder/'metadata.parquet')
    with np.load(folder/'windows_raw.npz') as data:
        X=data['X']; M=data['observed_mask']
        assert X.shape==(len(metadata),WINDOW_DAYS,len(FEATURES))
        assert M.shape==X.shape and M.all()
        assert np.isfinite(X).all() and (X>=0).all()
    with np.load(folder/'windows_scaled.npz') as data:
        assert data['X'].shape==X.shape and np.isfinite(data['X']).all()
    assert not metadata.duplicated(['serial_number','end_date']).any()
    assert set(metadata['serial_number']).issubset(set(ids))
    assert metadata['end_date'].between(*PERIODS[split]).all()
    assert metadata['end_date'].sub(metadata['start_date']).dt.days.eq(WINDOW_DAYS-1).all()
    checks.append({'split':split,'status':'passed'})
assert max(pd.Timestamp(PERIODS[s][1]) for s in ['train','calibration','development']) < pd.Timestamp(PERIODS['test'][0])
provenance={'created_utc':datetime.now(timezone.utc).isoformat(),
    'config_sha256':sha256(CONFIG_PATH),'split_sha256':sha256(SPLIT_PATH),
    'preprocessing_sha256':sha256(PREPROCESS_PATH),
    'archives':{q:{'filename':p.name,'bytes':p.stat().st_size,'sha256':sha256(p)} for q,p in ARCHIVES.items()},
    'checks':checks,'failure_labels_used_for_fitting':False}
(RUN/'dataset_provenance.json').write_text(json.dumps(provenance,indent=2))
# Record working-file integrity so Notebook 03 cannot silently mix array versions.
working_hashes={str(p.relative_to(COMPONENT_ROOT)):sha256(p)
    for split in ['train','calibration','development']
    for p in (FEATURE_ROOT/split).iterdir() if p.is_file()}
(RUN/'dataset_complete.json').write_text(json.dumps({
    'run_id':RUN_ID,'status':'passed','config_sha256':sha256(CONFIG_PATH),
    'split_sha256':sha256(SPLIT_PATH),'working_hashes':working_hashes},indent=2))
print('NOTEBOOK 02 COMPLETE — all saved-data checks passed.')
print('Next: open HDD_03_Baseline_Comparison.ipynb and use the same COMPONENT_ROOT and RUN_ID.')


NOTEBOOK 02 COMPLETE — all saved-data checks passed.
Next: open HDD_03_Baseline_Comparison.ipynb and use the same COMPONENT_ROOT and RUN_ID.


In [13]:
print("COMPONENT_ROOT:", COMPONENT_ROOT.resolve())
print("RUN_ID:", RUN_ID)

completion_file = (
    COMPONENT_ROOT / "experiments" / RUN_ID / "dataset_complete.json"
)

print("Completion file:", completion_file.resolve())
print("File exists:", completion_file.exists())

COMPONENT_ROOT: D:\SLIIT\Y4S1\RP\Research\J26-DS-342-\01_component_1_behavioral_drift_genai
RUN_ID: single_model_q1q2_2021_v1
Completion file: D:\SLIIT\Y4S1\RP\Research\J26-DS-342-\01_component_1_behavioral_drift_genai\experiments\single_model_q1q2_2021_v1\dataset_complete.json
File exists: True
